# Project of Brazilian Stocks Prediction

---

## ETL

---

## Imports

In [ ]:
import os
import pandas as pd
import numpy as np

#import streamlit as st

from tqdm import tqdm
import gc

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error

from materials_stock_libs.utils import (
    generate_future_dates, 
    contagem_frequencia_histograma,
    normalizar_codigos,
    preparar_chave,
    merge_grande,
    unicos,
    pivotar_por_categoria
    )

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 100)

import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

## Importing environment variables

In [ ]:
ROOT_PATH = os.path.join(os.getcwd(), '..')
DATA_PATH = os.path.join(ROOT_PATH, 'data')
OUTPUTS_PATH = os.path.join(ROOT_PATH, 'outputs')

In [ ]:
np.random.seed(42)

## Data reading

In [ ]:
# --- Filtra apenas o ano desejado ---
df_ext_pivot_2025 = df_ext_pivot[df_ext_pivot['DT_EMISSAO'].dt.year == 2025].copy()

# --- Agrupa por produto e calcula as métricas ---
pareto_df = (
    df_ext_pivot_2025
    .groupby('PRODUCT_DESC', as_index=False)
    .agg({
        'QUANTIDADE': 'sum',
        'RS_PPP': 'sum'
    })
    .sort_values('QUANTIDADE', ascending=False)
    .reset_index(drop=True)
)

# --- Calcula % acumulado por quantidade ---
pareto_df['%_ACUMULADO_QTD'] = pareto_df['QUANTIDADE'].cumsum() / pareto_df['QUANTIDADE'].sum() * 100

# --- Calcula % acumulado por valor (RS_PPP) ---
pareto_df['%_ACUMULADO_RS_PPP'] = pareto_df['RS_PPP'].cumsum() / pareto_df['RS_PPP'].sum() * 100

# --- Seleciona os produtos que compõem até 80% das quantidades ---
top_n = len(pareto_df[pareto_df['%_ACUMULADO_QTD'] <= 80])
top_products = pareto_df.head(top_n)['PRODUCT_DESC'].tolist()

# --- Exibe resumo ---
print(f"Total de produtos no Pareto (80% das quantidades): {len(top_products)}")
print(
    pareto_df.head(top_n)
    .assign(
        QUANTIDADE=lambda x: x['QUANTIDADE'].apply(lambda v: f"{v:,.0f}".replace(",", ".")),
        RS_PPP=lambda x: x['RS_PPP'].apply(lambda v: f"R$ {v:,.0f}".replace(",", ".")),
        **{'%_ACUMULADO_QTD': lambda x: x['%_ACUMULADO_QTD'].round(2),
           '%_ACUMULADO_RS_PPP': lambda x: x['%_ACUMULADO_RS_PPP'].round(2)}
    )
)

In [ ]:
pareto_df_filtered = pareto_df.head(top_n)
pareto_df_filtered['RS_PPP'].sum()

Barplot: mostra a frequencia e participação das categorias por um determinado período.

In [ ]:
def plot_top_n_barplot(
    df,
    cols_group=['UF', "EAN"],
    col_value='RS_PPP',
    date_col=None,
    ano=None,
    top_n_dim=10,
    top_n_subdim=5,
    title=None,
    ylabel=None,
    xlabel=None
):
    """
    Gráfico interativo de barras com Plotly, mostrando os Top N produtos (EAN)
    por participação dentro das Top N categorias (UF, Cidade, etc.).
    
    Adaptação:
    - Permite filtrar o dataframe por ano (coluna de data + ano desejado).
    - Ideal para uso em Streamlit.
    """

    df = df.copy()

    # --- Se o usuário informar uma coluna de data e ano, aplica o filtro ---
    if date_col is not None and ano is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col].dt.year == int(ano)]
        if df.empty:
            raise ValueError(f"Nenhum dado encontrado para o ano {ano} na coluna '{date_col}'.")

    # --- Agrupa e calcula soma ---
    df_grouped = df.groupby(cols_group, as_index=False)[col_value].sum()

    # --- Seleciona as top categorias principais (ex: UFs) ---
    top_dims = (
        df_grouped.groupby(cols_group[0])[col_value]
        .sum()
        .nlargest(top_n_dim)
        .index
    )
    df_grouped = df_grouped[df_grouped[cols_group[0]].isin(top_dims)]

    # --- Calcula participação (%) dentro de cada grupo principal ---
    df_grouped['share'] = (
        df_grouped.groupby(cols_group[0])[col_value]
        .transform(lambda x: x / x.sum())
        * 100
    )

    # --- Seleciona os top subgrupos (ex: EANs) dentro de cada grupo principal ---
    top_data = (
        df_grouped.sort_values([cols_group[0], 'share'], ascending=[True, False])
        .groupby(cols_group[0])
        .head(top_n_subdim)
    )

    # --- Criação do gráfico Plotly ---
    fig = px.bar(
        top_data,
        x=cols_group[1],
        y='share',
        color=cols_group[0],
        text='share',
        barmode='group',
        labels={
            'share': ylabel or 'Participação (%)',
            cols_group[1]: xlabel or cols_group[1],
            cols_group[0]: cols_group[0],
        },
        title=title or (
            f"Top {top_n_subdim} {cols_group[1]} "
            f"por participação nas {top_n_dim} maiores {cols_group[0]}"
            + (f" — Ano {ano}" if ano else "")
        ),
    )

    # --- Ajustes visuais ---
    fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
    fig.update_layout(
        xaxis_tickangle=-45,
        plot_bgcolor='white',
        yaxis=dict(showgrid=True, gridcolor='lightgray'),
        legend_title_text=cols_group[0],
        title_x=0.5,
        height=600,
    )

    return fig

fig = plot_top_n_barplot(
    df_ext_pivot,
    cols_group=['PRODUCT_DESC', 'UF'],
    col_value='RS_PPP',
    # date_col='DT_EMISSAO',
    # ano=2025,
    top_n_dim=5,
    top_n_subdim=5,
    title="Top 5 produtos por participação dentro das 5 maiores UFs",
    ylabel="Participação (%)",
    xlabel="Produto"
)

fig.show()
#st.plotly_chart(fig, use_container_width=True)

Treemap: Show the distribution of stocks by sector

In [ ]:
def plot_treemap(df, group_cols=['UF', 'SEGMENTO_PROD'], col_value='RS_PPP', date_col=None, ano=None):
    """Treemap para análise de participação de mercado (market share)."""
    df = df.copy()
    if date_col and ano:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col].dt.year == int(ano)]

    df_treemap = df.groupby(group_cols, as_index=False)[col_value].sum()
    fig = px.treemap(
        df_treemap,
        path=group_cols,
        values=col_value,
        color=col_value,
        color_continuous_scale='Viridis',
        title=f"Market Share de Receita — {ano if ano else 'Geral'} (até maio/2025)"
    )
    return fig

fig_treemap = plot_treemap(df_ext_pivot, group_cols=['UF'], col_value='RS_PPP', date_col='DT_EMISSAO', ano=2025)
fig_treemap.show()
#st.plotly_chart(fig_treemap, use_container_width=True)

### Concentration Analysis - Distribution of stocks by sector

**Objective**: measure if a few brands or products dominate the market.

In [ ]:
def plot_pareto_chart(df, group_col='EAN', col_value='RS_PPP', date_col=None, ano=None):
    """Curva de Pareto (Distribuição acumulada das vendas por EAN) com eixo duplo (80/20)."""
    df = df.copy()
    if date_col and ano:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col].dt.year == int(ano)]

    df_pareto = df.groupby(group_col)[col_value].sum().sort_values(ascending=False)
    df_cum = df_pareto.cumsum() / df_pareto.sum() * 100

    fig = go.Figure()

    fig.add_bar(
        x=df_pareto.index.astype(str),
        y=df_pareto.values,
        name='Vendas',
        marker_color='royalblue',
        yaxis='y1'
    )

    fig.add_trace(go.Scatter(
        x=df_pareto.index.astype(str),
        y=df_cum,
        mode='lines+markers',
        name='Cumulativo (%)',
        marker_color='red',
        yaxis='y2'
    ))

    fig.add_hline(y=80, line_dash='dash', line_color='red', annotation_text='80%', yref='y2')

    fig.update_layout(
        title=f"Pareto Chart — {ano if ano else 'Geral'}",
        xaxis=dict(title=group_col, tickangle=45),
        yaxis=dict(title=col_value, showgrid=False),
        yaxis2=dict(
            title='Cumulativo (%)',
            overlaying='y',
            side='right',
            range=[0, 110]
        ),
        legend=dict(x=0.80, y=0.02),
        template='plotly_white',
        height=550,
        margin=dict(l=50, r=50, b=100, t=80)
    )
    return fig

fig_pareto = plot_pareto_chart(df_ext_pivot, group_col='PRODUCT_DESC', col_value='RS_PPP', date_col='DT_EMISSAO', ano=2025)
fig_pareto.show()
#st.plotly_chart(fig_pareto, use_container_width=True)

KDE: Show the distribution of stock prices

In [ ]:
import plotly.figure_factory as ff

def plot_kde_participacao(df, ean_col='EAN', col_value='RS_PPP', date_col=None, ano=None):
    """Distribuição (KDE) da participação de vendas entre EANs."""
    df = df.copy()
    if date_col and ano:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col].dt.year == int(ano)]

    vendas = df.groupby(ean_col)[col_value].sum()
    fig = ff.create_distplot([vendas.values], [f"Distribuição {col_value}"], show_hist=False)
    fig.update_layout(
        title=f"Distribuição de vendas por {ean_col} — {ano if ano else 'Geral'}",
        xaxis_title=col_value,
        yaxis_title='Densidade',
        height=500,
        template='plotly_white'
    )
    return fig

fig_kde = plot_kde_participacao(df_ext_pivot, ean_col='PRODUCT_DESC', col_value='RS_PPP', date_col='DT_EMISSAO', ano=2025)
fig_kde.show()
#st.plotly_chart(fig_kde, use_container_width=True)

### Comparação Cruzada

**Objetivo**: entender se há padrões ou afinidades entre variáveis categóricas (ex: produto × UF).

Heatmap: Mostra concentração de vendas por meio de uma correlação entre 2 features.

In [ ]:
def plot_heatmap(df, prod_col='PRODUCT_DESC', uf_col='UF', col_value='RS_PPP', date_col=None, ano=None):
    """Heatmap cruzando Produto x UF (intensidade = vendas)."""
    df = df.copy()
    if date_col and ano:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col].dt.year == int(ano)]

    pivot = df.pivot_table(values=col_value, index=prod_col, columns=uf_col, aggfunc='sum', fill_value=0)
    fig = px.imshow(
        pivot,
        labels=dict(x=uf_col, y=prod_col, color=col_value),
        title=f"Heatmap de Vendas — {ano if ano else 'Geral'}",
        aspect='auto',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=700)
    return fig

fig_heatmap = plot_heatmap(df_ext_pivot, prod_col='PRODUCT_DESC', uf_col='UF', col_value='QUANTIDADE', date_col='DT_EMISSAO', ano=2025)
fig_heatmap.show()
#st.plotly_chart(fig_heatmap, use_container_width=True)

Clustered barplot: Compara grupos de categorias. Útil para resumo executivo.

In [ ]:
def plot_clustered_barplot(df, group_col='UF', category_col='SEGMENTO_PROD', col_value='RS_PPP', date_col=None, ano=None):
    """Clustered barplot para comparar categorias dentro de grupos."""
    df = df.copy()
    if date_col and ano:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col].dt.year == int(ano)]

    grouped = df.groupby([group_col, category_col])[col_value].sum().reset_index()
    fig = px.bar(
        grouped,
        x=group_col,
        y=col_value,
        color=category_col,
        barmode='group',
        title=f"Comparação de categorias por {group_col} ({ano if ano else 'Geral'})",
        labels={col_value: 'Vendas', group_col: group_col, category_col: category_col},
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=600, template='plotly_white')
    return fig

fig_clustered_barplot = plot_clustered_barplot(df_ext_pivot, group_col='PRODUCT_DESC', category_col='UF', col_value='RS_PPP', date_col='DT_EMISSAO', ano=2025)
fig_clustered_barplot.show()
#st.plotly_chart(fig_clustered_barplot, use_container_width=True)

Bubble chart: Relações multivariadas visuais, pode mostrar outliers e dominância.

In [ ]:
def plot_bubble_chart(df, ean_col='EAN', seg_col='SEGMENTO_PROD', col_qty='QUANTIDADE', col_value='RS_PPP', date_col=None, ano=None):
    """Bubble Chart: EAN × Segmento (tamanho = QUANTIDADE, cor = RS_PPP)."""
    df = df.copy()
    if date_col and ano:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df[df[date_col].dt.year == int(ano)]

    bubble_data = df.groupby([ean_col, seg_col], as_index=False).agg({
        col_qty: 'sum',
        col_value: 'sum'
    })
    bubble_data[ean_col] = bubble_data[ean_col].astype(str)

    fig = px.scatter(
        bubble_data,
        x=ean_col,
        y=seg_col,
        size=col_qty,
        color=col_value,
        hover_data=[ean_col, seg_col, col_qty, col_value],
        title=f"Bubble Chart: {ean_col} × {seg_col} ({ano if ano else 'Geral'})",
        size_max=60,
        color_continuous_scale='Viridis'
    )
    fig.update_layout(
        xaxis_tickangle=45,
        height=600,
        template='plotly_white'
    )
    return fig

fig_bubble_chart = plot_bubble_chart(df_ext_pivot, ean_col='PRODUCT_DESC', seg_col='UF', col_qty='QUANTIDADE', col_value='RS_PPP', date_col='DT_EMISSAO', ano=2025)
fig_bubble_chart.show()
#st.plotly_chart(fig_bubble_chart, use_container_width=True)